# 1단계: Korean sLLM 일반 텍스트 사전학습 (Colab)

기존 `ModelConfig`(24L/d768, vocab 32768, MTP 16)를 그대로 사용해 랜덤 초기화부터 학습합니다.
기존 SentencePiece 토크나이저를 사용하며 이후 SFT에서도 동일 파일을 사용해야 합니다.

1. Colab에서 GPU 런타임을 선택하고 아래 셀을 순서대로 실행하세요.
2. Drive의 `korean_sllm_data/pretrain/pretrain_train.json`, `pretrain_val.json`에 일반 텍스트를 준비하세요.
   파일은 `["한국어 문서 본문...", "다음 문서..."]` 형식의 JSON 문자열 배열입니다. 채팅 템플릿이나 user/assistant 필드는 사용하지 않습니다.
   JSON 문자열의 줄바꿈은 `\n`으로 이스케이프하세요. val은 문서 단위로 분리하고 train 중복을 제거하세요.
   코퍼스는 사용자가 준비합니다. 아래 예시 문서는 형식 설명용이며 학습 데이터로 자동 사용하지 않습니다.
   체크포인트(`best.pt`, `last.pt`)와 토크나이저(`spm.model`)도 같은 `pretrain` 폴더에 저장합니다.
3. 학습이 끝나면 `colab_train_think_weight_01.ipynb`로 인스트럭션 학습을 진행하세요.

문서마다 BOS/EOS를 붙이고 모든 다음 토큰에 weight=1로 main CE와 MTP CE를 계산합니다.
긴 문서를 길이 제한으로 버리지 않고 이어 붙여 패킹합니다. 문서 간 attention은 허용하며 EOS로 경계를 표시합니다.
윈도우는 2048토큰, 겹침은 256토큰(이동 간격 1792)입니다.
마지막 윈도우는 코퍼스 끝에 맞춰 겹침을 늘려 남는 토큰도 포함합니다.
전체 코퍼스가 2048토큰보다 짧으면 그 길이 그대로 사용하며 패딩하지 않습니다.
마지막 작은 배치도 학습하고 epoch 스텝은 올림 계산하여 전체 데이터가 포함됩니다. MTP는 각 윈도우 안에서만 예측합니다.
캐시는 원문+토크나이저 해시로 구분하고 문서 단위로 디스크에 기록합니다(전체 코퍼스를 RAM에 올리지 않음).

기본값은 기존 노트북과 같은 **96GB GPU / seq 2048 / batch 8 / accum 4** 기준입니다.
T4 등에서는 batch를 낮추고 `GRAD_CHECKPOINTING=True`로 조정해야 하며, 메모리 적합성은 실행해서 확인해야 합니다.
사전학습에는 충분한 양의 중복 제거된 한국어 코퍼스가 필요합니다. epochs=1, lr=3e-4는 시작값이며 성능 개선을 보장하지 않습니다.


In [ ]:
# 1) 리포 준비 (Colab GPU 런타임을 먼저 선택하세요)
from pathlib import Path
import subprocess
import os
REPO_URL = "https://github.com/MinsuChae/korean_sllm.git"
REPO_DIR = Path('/content/korean_sllm')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt', 'protobuf>=4.25.0', 'ijson>=3.2.0'], check=True)


In [ ]:
# 이 노트북에 포함된 사전학습 코드를 사용합니다.
from pathlib import Path
Path('train.py').write_text('"""학습 스크립트 (Colab / 로컬 공용).\n\n  python train.py                          # base 프리셋 (24L, d768), GPU 권장\n  python train.py --preset tiny --max-steps 30   # 로컬 smoke test\n\n- pretrain_train.json / pretrain_val.json의 문자열 전체를 weight=1로 학습한다.\n- bf16 지원 GPU 는 bf16 autocast, 그 외 CUDA 는 fp16+GradScaler, CPU 는 fp32.\n- --ckpt-dir 에 last.pt(--save-every 주기 최신)와 best.pt(val main_loss 최저)만 유지하고,\n  --resume 은 last.pt 로 재개한다 (Google Drive 경로 가능).\n"""\n\nimport argparse\nimport hashlib\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom data import load_tokenizer\nfrom model import KoreanSLLM, ModelConfig\n\nROOT = Path(__file__).resolve().parent\n\nPRESETS = {\n    "base": {},\n    "tiny": {"n_layers": 2, "d_model": 64, "n_heads": 4, "n_kv_heads": 2, "head_dim": 16,\n             "ffn_hidden": 128, "mtp_ffn_hidden": 64, "max_seq_len": 128, "sliding_window": 32},\n}\n\n\ndef parse_args():\n    p = argparse.ArgumentParser(description="Korean sLLM 학습")\n    p.add_argument("--preset", choices=PRESETS, default="base")\n    p.add_argument("--data-dir", default=str(ROOT))\n    p.add_argument("--cache-dir", default=str(ROOT / "cache"))\n    p.add_argument("--ckpt-dir", default=str(ROOT / "checkpoints"))\n    p.add_argument("--stage", choices=("pretrain",), default="pretrain")\n    p.add_argument("--overlap", type=int, default=256)\n    checkpoint = p.add_mutually_exclusive_group()\n    checkpoint.add_argument("--resume", default=None, help="동일 단계의 모델/옵티마이저/스텝 복원")\n    checkpoint.add_argument("--init-from", default=None, help="사전학습 모델 가중치만 복원; 새 스케줄 시작")\n    p.add_argument("--seq-len", type=int, default=None, help="기본: config.max_seq_len")\n    p.add_argument("--batch-size", type=int, default=8)\n    p.add_argument("--grad-accum", type=int, default=4)\n    p.add_argument("--max-steps", type=int, default=20_000)\n    p.add_argument("--epochs", type=float, default=None,\n                   help="지정 시 max-steps 를 무시하고 데이터셋 윈도우 수에서 스텝을 환산 (권장: 4)")\n    p.add_argument("--lr", type=float, default=3e-4)\n    p.add_argument("--min-lr-ratio", type=float, default=0.1)\n    p.add_argument("--warmup-steps", type=int, default=500)\n    p.add_argument("--weight-decay", type=float, default=0.1)\n    p.add_argument("--grad-clip", type=float, default=1.0)\n    p.add_argument("--eval-every", type=int, default=500)\n    p.add_argument("--eval-batches", type=int, default=50)\n    p.add_argument("--save-every", type=int, default=1000)\n    p.add_argument("--log-every", type=int, default=20)\n    p.add_argument("--grad-checkpointing", action="store_true")\n    p.add_argument("--num-workers", type=int, default=2)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--sample", default=None, help="학습 종료 후 이 프롬프트로 생성 데모")\n    return p.parse_args()\n\n\ndef lr_at(step: int, args) -> float:\n    if step < args.warmup_steps:\n        return args.lr * (step + 1) / args.warmup_steps\n    t = (step - args.warmup_steps) / max(args.max_steps - args.warmup_steps, 1)\n    return args.lr * (args.min_lr_ratio + (1 - args.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * min(t, 1.0))))\n\n\ndef setup_amp(device: torch.device):\n    if device.type == "cuda" and torch.cuda.is_bf16_supported():\n        return torch.bfloat16, None\n    if device.type == "cuda":\n        return torch.float16, torch.amp.GradScaler("cuda")\n    return None, None  # CPU: fp32\n\n\n@torch.no_grad()\ndef evaluate(model, loader, device, amp_dtype, max_batches: int) -> dict[str, float]:\n    model.eval()\n    sums = {"main_loss": 0.0, "mtp_loss": 0.0}\n    n = 0\n    for batch in loader:\n        if n >= max_batches:\n            break\n        ids = batch["input_ids"].to(device)\n        mask = batch["loss_mask"].to(device)\n        with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):\n            out = model(ids, mask)\n        sums["main_loss"] += out["main_loss"].item()\n        sums["mtp_loss"] += out["mtp_loss"].item()\n        n += 1\n    model.train()\n    return {k: v / max(n, 1) for k, v in sums.items()}\n\n\ndef save_ckpt(path: Path, model, optim, step: int, cfg: ModelConfig, best_val: float,\n              stage: str, tokenizer_sha256: str, scaler=None):\n    # 임시 파일에 쓴 뒤 교체 - Drive 위에서 덮어쓰기 도중 런타임이 끊겨도 기존 파일이 보존된다\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(".tmp")\n    torch.save({"model": model.state_dict(), "optim": optim.state_dict(),\n                "step": step, "config": cfg.to_dict(), "best_val": best_val,\n                "stage": stage, "tokenizer_sha256": tokenizer_sha256,\n                "scaler": scaler.state_dict() if scaler else None}, tmp)\n    os.replace(tmp, path)\n    print(f"[ckpt] step {step} -> {path}")\n\n\ndef main():\n    args = parse_args()\n    torch.manual_seed(args.seed)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n    if args.init_from and args.stage != "sft":\n        raise ValueError("--init-from은 pretrain -> sft 전환에 사용하세요.")\n    ckpt = None\n    checkpoint_path = args.resume or args.init_from\n    if checkpoint_path:\n        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n        expected_stage = "pretrain" if args.init_from else args.stage\n        if ckpt.get("stage", "sft") != expected_stage:\n            raise ValueError("체크포인트 단계 불일치: pretrain -> sft에는 --init-from을 사용하세요.")\n        if args.init_from and Path(args.ckpt_dir).resolve() == Path(args.init_from).resolve().parent:\n            raise ValueError("SFT 출력 폴더는 사전학습 체크포인트 폴더와 분리하세요.")\n    cfg = ModelConfig(**ckpt["config"]) if ckpt else ModelConfig(**PRESETS[args.preset])\n    seq_len = args.seq_len or cfg.max_seq_len\n    if not 2 <= seq_len <= cfg.max_seq_len:\n        raise ValueError(f"seq-len은 2..{cfg.max_seq_len} 범위여야 합니다.")\n    if min(args.batch_size, args.grad_accum, args.eval_every, args.save_every,\n           args.eval_batches, args.log_every, args.max_steps) < 1:\n        raise ValueError("배치/스텝/주기 값은 양수여야 합니다.")\n    if args.epochs is not None and args.epochs <= 0:\n        raise ValueError("epochs는 양수여야 합니다.")\n    root, cache_dir, ckpt_dir = Path(args.data_dir), Path(args.cache_dir), Path(args.ckpt_dir)\n\n    sp = load_tokenizer()\n    tokenizer_sha256 = hashlib.sha256(sp.serialized_model_proto()).hexdigest()\n    if ckpt:\n        saved_hash = ckpt.get("tokenizer_sha256")\n        if args.init_from and not saved_hash:\n            raise ValueError("토크나이저 해시가 있는 새 pretrain 체크포인트가 필요합니다.")\n        if saved_hash and saved_hash != tokenizer_sha256:\n            raise ValueError("토크나이저가 체크포인트와 다릅니다. 사전학습 때의 spm.model을 사용하세요.")\n    assert sp.get_piece_size() == cfg.vocab_size, \\\n        f"토크나이저 vocab {sp.get_piece_size()} != config {cfg.vocab_size}"\n\n    from pretrain_data import make_pretrain_dataset\n    train_ds = make_pretrain_dataset(root, "train", seq_len, cache_dir, sp, args.overlap)\n    val_ds = make_pretrain_dataset(root, "val", seq_len, cache_dir, sp, args.overlap)\n    if len(train_ds) == 0 or len(val_ds) == 0:\n        raise ValueError("학습 또는 검증 데이터가 비었습니다.")\n    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,\n                              num_workers=args.num_workers, drop_last=False, pin_memory=device.type == "cuda")\n    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,\n                            num_workers=args.num_workers, drop_last=False)\n\n    # 토크나이저·seq_len 이 바뀌면 1 epoch 스텝 수가 달라지므로 epoch 로 지정할 수 있게 한다\n    steps_per_epoch = max(math.ceil(len(train_loader) / args.grad_accum), 1)\n    if args.epochs is not None:\n        args.max_steps = max(math.ceil(args.epochs * steps_per_epoch), 1)\n    tokens_per_step = args.batch_size * args.grad_accum * seq_len\n\n    model = KoreanSLLM(cfg).to(device)\n    print(f"모델: {cfg.n_layers}L d{cfg.d_model} v{cfg.vocab_size} | 파라미터 {model.num_params() / 1e6:.1f}M | "\n          f"MTP {cfg.mtp_n}토큰 | device={device} | train {len(train_ds):,} windows(seq {seq_len})")\n    print(f"스케줄: {args.max_steps:,} steps × {tokens_per_step:,} tok/step "\n          f"= {args.max_steps * tokens_per_step / 1e9:.2f}B tokens ≈ {args.max_steps / steps_per_epoch:.1f} epochs")\n\n    if args.grad_checkpointing:\n        import functools\n        from torch.utils.checkpoint import checkpoint as checkpoint_forward\n        for block in model.blocks:\n            block._orig_forward = block.forward\n            block.forward = functools.partial(\n                checkpoint_forward, block._orig_forward, use_reentrant=False)\n\n    decay, no_decay = [], []\n    for name, param in model.named_parameters():\n        (no_decay if param.ndim < 2 else decay).append(param)\n    optim = torch.optim.AdamW(\n        [{"params": decay, "weight_decay": args.weight_decay},\n         {"params": no_decay, "weight_decay": 0.0}],\n        lr=args.lr, betas=(0.9, 0.95), fused=device.type == "cuda")\n\n    amp_dtype, scaler = setup_amp(device)\n    print(f"정밀도: {amp_dtype or torch.float32}")\n\n    step = 0\n    best_val = float("inf")\n    if ckpt:\n        model.load_state_dict(ckpt["model"])\n    if args.init_from:\n        print(f"[init] {args.init_from}: 모델 구조/가중치 복원, optimizer/step/best_val 초기화")\n    if args.resume:\n        optim.load_state_dict(ckpt["optim"])\n        if scaler and ckpt.get("scaler"):\n            scaler.load_state_dict(ckpt["scaler"])\n        step = ckpt["step"]\n        best_val = ckpt.get("best_val", float("inf"))  # 미복원 시 첫 eval 이 best.pt 를 덮어쓴다\n        print(f"[ckpt] {args.resume} 에서 step {step} 재개 (best val {best_val:.4f})")\n    del ckpt\n\n    model.train()\n    data_iter = iter(train_loader)\n    t0, tokens_seen = time.time(), 0\n    while step < args.max_steps:\n        for g in optim.param_groups:\n            g["lr"] = lr_at(step, args)\n        optim.zero_grad(set_to_none=True)\n        logs = {"loss": 0.0, "main_loss": 0.0, "mtp_loss": 0.0}\n        for _ in range(args.grad_accum):\n            try:\n                batch = next(data_iter)\n            except StopIteration:\n                data_iter = iter(train_loader)\n                batch = next(data_iter)\n            ids = batch["input_ids"].to(device)\n            mask = batch["loss_mask"].to(device)\n            with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):\n                out = model(ids, mask)\n            loss = out["loss"] / args.grad_accum\n            (scaler.scale(loss) if scaler else loss).backward()\n            for k in logs:\n                logs[k] += out[k].item() / args.grad_accum\n            tokens_seen += ids.numel()\n\n        if scaler:\n            scaler.unscale_(optim)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)\n        if scaler:\n            scaler.step(optim)\n            scaler.update()\n        else:\n            optim.step()\n        step += 1\n\n        if step % args.log_every == 0:\n            tps = tokens_seen / (time.time() - t0)\n            print(f"step {step:6d} | loss {logs[\'loss\']:.4f} (main {logs[\'main_loss\']:.4f} "\n                  f"mtp {logs[\'mtp_loss\']:.4f}) | lr {optim.param_groups[0][\'lr\']:.2e} | {tps / 1e3:.1f}k tok/s")\n        if step % args.eval_every == 0 or step == args.max_steps:\n            ev = evaluate(model, val_loader, device, amp_dtype, args.eval_batches)\n            mem = (f" | mem {torch.cuda.max_memory_allocated() / 2**30:.1f}GiB"\n                   if device.type == "cuda" else "")\n            print(f"  [val] main {ev[\'main_loss\']:.4f} | mtp {ev[\'mtp_loss\']:.4f} | "\n                  f"ppl {math.exp(min(ev[\'main_loss\'], 20)):.1f}{mem}")\n            if ev["main_loss"] < best_val:\n                best_val = ev["main_loss"]\n                print(f"  [ckpt] new best (val main {best_val:.4f})")\n                save_ckpt(ckpt_dir / "best.pt", model, optim, step, cfg, best_val,\n                          args.stage, tokenizer_sha256, scaler)\n        if step % args.save_every == 0 or step == args.max_steps:\n            save_ckpt(ckpt_dir / "last.pt", model, optim, step, cfg, best_val,\n                      args.stage, tokenizer_sha256, scaler)\n\n    print(f"학습 종료: last.pt step {step} | best.pt val main {best_val:.4f}")\n\n    if args.sample:\n        prompt_ids = [sp.bos_id()] + sp.encode(args.sample)\n        out = model.generate(torch.tensor([prompt_ids], device=device), max_new_tokens=256)\n        print("\\n=== 생성 데모 ===")\n        print(sp.decode(out[0].tolist()))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('pretrain_data.py').write_text('"""Plain-text JSON string array -> disk-backed causal-LM windows (no chat template)."""\n\nimport hashlib\nimport ijson\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\n\nclass PretrainDataset(Dataset):\n    def __init__(self, path: Path, seq_len: int, overlap: int = 256):\n        if not 1 <= overlap < seq_len:\n            raise ValueError("overlap은 1 이상 seq_len 미만이어야 합니다.")\n        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")\n        self.seq_len = seq_len\n        self.stride = seq_len - overlap\n\n    def __len__(self):\n        remaining = max(0, len(self.tokens) - self.seq_len)\n        return 1 + (remaining + self.stride - 1) // self.stride\n\n    def __getitem__(self, index):\n        if not 0 <= index < len(self):\n            raise IndexError(index)\n        # End-align the final window: retain every token without padding.\n        # Only the final overlap may exceed the configured overlap.\n        start = min(index * self.stride, max(0, len(self.tokens) - self.seq_len))\n        ids = torch.from_numpy(self.tokens[start:start + self.seq_len].astype(np.int64))\n        return {"input_ids": ids, "loss_mask": torch.ones_like(ids)}\n\n\ndef iter_texts(source):\n    with Path(source).open("rb") as stream:\n        events = ijson.parse(stream)\n        if next(events, None) != ("", "start_array", None):\n            raise ValueError(f"{source}: JSON 문자열 배열이 필요합니다.")\n        for prefix, event, value in events:\n            if prefix == "" and event == "end_array":\n                # Consume the parser to reject trailing invalid JSON as well.\n                if next(events, None) is not None:\n                    raise ValueError(f"{source}: 배열 뒤에 데이터가 있습니다.")\n                return\n            if prefix != "item" or event != "string":\n                raise ValueError(f"{source}: 모든 배열 원소는 문자열이어야 합니다.")\n            yield value\n        raise ValueError(f"{source}: 배열이 완성되지 않았습니다.")\n\n\ndef make_pretrain_dataset(root, split, seq_len, cache_dir, sp, overlap=256):\n    source = Path(root) / f"pretrain_{split}.json"\n    if not source.is_file():\n        raise FileNotFoundError(f\'{source}: ["본문", ...] 형식의 JSON 문자열 배열을 준비하세요.\')\n    if not 0 <= sp.bos_id() < sp.get_piece_size() or not 0 <= sp.eos_id() < sp.get_piece_size():\n        raise ValueError("BOS/EOS가 있는 기존 토크나이저가 필요합니다.")\n    if sp.get_piece_size() > 65536:\n        raise ValueError("uint16 vocabulary overflow")\n    digest = hashlib.sha256(sp.serialized_model_proto())\n    with source.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    target = cache_dir / f"pretrain_v2_json_{split}_{digest.hexdigest()}.bin"\n    if not target.exists():\n        tmp = target.with_suffix(".tmp")\n        documents = tokens = 0\n        try:\n            with tmp.open("wb") as out:\n                for text in iter_texts(source):\n                    ids = [sp.bos_id()] + sp.encode(text) + [sp.eos_id()]\n                    np.asarray(ids, dtype=np.uint16).tofile(out)\n                    documents += 1\n                    tokens += len(ids)\n            if tokens == 0:\n                raise ValueError(f"{source}: 비어 있는 코퍼스")\n            os.replace(tmp, target)\n        finally:\n            tmp.unlink(missing_ok=True)\n        print(f"[pretrain] {split}: {documents:,} documents / {tokens:,} tokens")\n    dataset = PretrainDataset(target, seq_len, overlap)\n    if len(dataset) == 0:\n        raise ValueError(f"{source}: 최소 {seq_len} tokens가 필요합니다.")\n    return dataset\n', encoding='utf-8')
print("학습 코드 준비 완료")


In [ ]:
# 2) Drive 및 데이터/출력 경로
from google.colab import drive
import shutil
import hashlib
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Colab 런타임 유형을 GPU로 변경하세요.'
DATA_DIR = Path('/content/drive/MyDrive/korean_sllm_data/pretrain')
CKPT_DIR = Path('/content/drive/MyDrive/korean_sllm_data/pretrain')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESUME = None  # 재개: str(CKPT_DIR / 'last.pt')
if RESUME:
    assert (CKPT_DIR / 'spm.model').is_file()
    shutil.copy2(CKPT_DIR / 'spm.model', 'tokenizer/spm.model')
else:
    assert not any(CKPT_DIR.glob('*.pt')), '기존 실행이 있습니다. RESUME을 지정하거나 새 CKPT_DIR을 사용하세요.'
    shutil.copy2('tokenizer/spm.model', CKPT_DIR / 'spm.model')
print('GPU:', torch.cuda.get_device_name())
print('tokenizer SHA256:', hashlib.sha256(Path('tokenizer/spm.model').read_bytes()).hexdigest())


In [ ]:
# 3) Drive 코퍼스를 Colab 로컬 디스크에 복사 (반복 학습 I/O 개선)
from pretrain_data import iter_texts
LOCAL_DATA = Path('/content/pretrain_data')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
for split in ('train', 'val'):
    source = DATA_DIR / f'pretrain_{split}.json'
    if not source.is_file():
        raise FileNotFoundError(f'{source}에 ["본문", ...] JSON 문자열 배열을 준비하세요.')
    local = LOCAL_DATA / source.name
    shutil.copy2(source, local)
    documents = iter_texts(local)
    try:
        assert next(documents, None) is not None, f'{source}: 비어 있는 코퍼스'
    finally:
        documents.close()
    print(split, 'bytes:', source.stat().st_size)


In [ ]:
# 4) 사전학습. 모델 구조는 변경하지 않습니다.
SEQ_LEN = 2048
OVERLAP = 256
BATCH_SIZE = 8
GRAD_ACCUM = 4
GRAD_CHECKPOINTING = False
EPOCHS = 1
cmd = [
    'python', 'train.py', '--stage', 'pretrain',
    '--data-dir', str(LOCAL_DATA), '--cache-dir', '/content/pretrain_cache',
    '--ckpt-dir', str(CKPT_DIR), '--seq-len', str(SEQ_LEN),
    '--overlap', str(OVERLAP),
    '--batch-size', str(BATCH_SIZE), '--grad-accum', str(GRAD_ACCUM),
    '--epochs', str(EPOCHS), '--lr', '3e-4', '--warmup-steps', '500',
    '--eval-every', '250', '--save-every', '500',
]
if GRAD_CHECKPOINTING:
    cmd.append('--grad-checkpointing')
if RESUME:
    cmd += ['--resume', RESUME]
subprocess.run(cmd, check=True)


In [ ]:
# 5) SFT로 전달할 파일 확인 및 Drive flush
# best.pt: 검증 main loss 최저 가중치 / last.pt: 사전학습 재개용
for filename in ('best.pt', 'last.pt', 'spm.model'):
    path = CKPT_DIR / filename
    assert path.is_file(), f'산출물 누락: {path}'
    print(path, path.stat().st_size, 'bytes')
drive.flush_and_unmount()
print('다음: colab_train_think_weight_01.ipynb에서 pretrain/best.pt로 SFT 시작')
